In [2]:
# Cell 1: Setup — imports, env config (.env), logging, OpenTelemetry
import sys, os, json, logging
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from dotenv import load_dotenv
load_dotenv()

from semantic_kernel import Kernel
from semantic_kernel.connectors.ai.open_ai import AzureChatCompletion
from azure.identity import DefaultAzureCredential, get_bearer_token_provider

from utils.logging_config import setup_logging, log_routing_decision, log_self_reflection
from utils.telemetry import setup_telemetry

# Structured JSON logger
logger = setup_logging(level=logging.INFO)

# OpenTelemetry tracer
tracer = setup_telemetry()

# Azure OpenAI config
AZURE_OPENAI_ENDPOINT = os.environ['AZURE_OPENAI_ENDPOINT']
AZURE_OPENAI_DEPLOYMENT = os.environ.get('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')
AZURE_OPENAI_API_VERSION = os.environ.get('AZURE_OPENAI_API_VERSION', '2024-12-01-preview')
AZURE_OPENAI_AUTH_MODE = os.environ.get('AZURE_OPENAI_AUTH_MODE', 'entra').lower()
AZURE_OPENAI_API_KEY = os.environ.get('AZURE_OPENAI_API_KEY')

if AZURE_OPENAI_AUTH_MODE not in {'entra', 'key'}:
    raise ValueError(f'Unsupported AZURE_OPENAI_AUTH_MODE: {AZURE_OPENAI_AUTH_MODE}')

def create_kernel() -> Kernel:
    kernel = Kernel()
    service_kwargs = {
        'service_id': 'azure_openai',
        'deployment_name': AZURE_OPENAI_DEPLOYMENT,
        'endpoint': AZURE_OPENAI_ENDPOINT,
        'api_version': AZURE_OPENAI_API_VERSION,
    }

    if AZURE_OPENAI_AUTH_MODE == 'entra':
        credential = DefaultAzureCredential()
        token_provider = get_bearer_token_provider(
            credential,
            'https://cognitiveservices.azure.com/.default',
        )
        service_kwargs['ad_token_provider'] = token_provider
    else:
        if not AZURE_OPENAI_API_KEY:
            raise ValueError('AZURE_OPENAI_API_KEY is required when AZURE_OPENAI_AUTH_MODE=key')
        service_kwargs['api_key'] = AZURE_OPENAI_API_KEY

    kernel.add_service(
        AzureChatCompletion(**service_kwargs)
    )
    return kernel

print(f'Setup complete')
print(f'  Endpoint: {AZURE_OPENAI_ENDPOINT[:50]}...')
print(f'  Deployment: {AZURE_OPENAI_DEPLOYMENT}')
print(f'  API version: {AZURE_OPENAI_API_VERSION}')
print(f'  Auth mode: {AZURE_OPENAI_AUTH_MODE}')
print(f'  SK version: {__import__("semantic_kernel").__version__}')

Overriding of current TracerProvider is not allowed


[Telemetry] ⚠️ azure-monitor-opentelemetry-exporter not installed. Using console exporter.
Setup complete
  Endpoint: https://dlstmvprtus-wingnut0310-ai.openai.azure.co...
  Deployment: gpt-4o
  API version: 2024-12-01-preview
  Auth mode: entra
  SK version: 1.41.2


In [3]:
# Cell 2: Define 3 dummy agents + load mock KB data
from agents.common.recommend_agent import RecommendPlugin
from agents.common.search_agent import SearchPlugin
from agents.common.policy_agent import PolicyPlugin

with open('mock_data/products.json', encoding='utf-8') as f:
    PRODUCTS = json.load(f)
with open('mock_data/search_index.json', encoding='utf-8') as f:
    SEARCH_INDEX = json.load(f)
with open('mock_data/policies.json', encoding='utf-8') as f:
    POLICIES = json.load(f)

rp = RecommendPlugin()
sp = SearchPlugin()
pp = PolicyPlugin()

print(f'Products catalog: {len(PRODUCTS)} items')
for p in PRODUCTS:
    stock = 'in stock' if p['in_stock'] else 'out of stock'
    print(f'  {p["name"]} ({p["brand"]}) - {p["price"]:,}won | {stock}')
print(f'\nSearch index: {len(SEARCH_INDEX)} entries')
print(f'Policy docs: {len(POLICIES)} documents')
print()
print('Demo queries:')
print('  1. 건성 피부에 좋은 크림 추천해줘')
print('  2. 설화수 자음생 크림 가격 알려줘')
print('  3. 교환 환불 정책 알려줘')
print('  4. (Ambiguous) 설화수 자음생 크림 건성 피부에 좋아?')


Products catalog: 5 items
  설화수 자음생 크림 (설화수) - 120,000won | in stock
  라네즈 워터뱅크 하이드로 크림 (라네즈) - 42,000won | in stock
  이니스프리 그린티 씨드 세럼 (이니스프리) - 29,000won | in stock
  헤라 블랙 쿠션 파운데이션 (헤라) - 55,000won | in stock
  마몽드 로즈워터 토너 (마몽드) - 18,000won | out of stock

Search index: 5 entries
Policy docs: 3 documents

Demo queries:
  1. 건성 피부에 좋은 크림 추천해줘
  2. 설화수 자음생 크림 가격 알려줘
  3. 교환 환불 정책 알려줘
  4. (Ambiguous) 설화수 자음생 크림 건성 피부에 좋아?


---
# SECTION 1a: BEFORE — Generic Black Box

**Single system-prompt completion. NO plugins, NO function calling.**

3개 역할을 system prompt에 직접 기술하고, 단일 LLM completion으로 응답합니다.  
어떤 역할이 응답을 처리했는지 **전혀 알 수 없는** 완전한 black box입니다.

| Routing Trace | Override | Logging Hook |
|:---:|:---:|:---:|
| ❌ None | ❌ No | ❌ No |


In [4]:
# Cell 4: Section 1a — Run black-box supervisor
from agents.before.blackbox_supervisor import run_blackbox_supervisor

before_kernel = create_kernel()

queries_1a = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 1a: Generic Black Box')
print('(NO plugins, NO function calling)')
print('=' * 60)

for q in queries_1a:
    print(f'\nQuery: {q}')
    response = await run_blackbox_supervisor(q, before_kernel)
    print(f'Response:\n{response[:500]}')
    print(f'\nRouting Trace: NONE')
    print(f'  -> No way to know which role handled this query')
    print('-' * 60)


SECTION 1a: Generic Black Box
(NO plugins, NO function calling)

Query: 건성 피부에 좋은 크림 추천해줘
Response:
건성 피부에는 수분을 깊게 공급하고 보습막을 형성해주는 크림이 좋습니다. 아래 제품들을 추천해드립니다:

1. **라로슈포제 시카플라스트 밤 B5**  
   - 시어버터와 판테놀 성분이 피부를 촉촉하고 편안하게 가꿔줍니다.

2. **클리니크 모이스처 써지 72시간 수분 크림**  
   - 히알루론산과 알로에 성분으로 보습 효과가 뛰어나고 산뜻한 마무리감을 제공합니다.

3. **아벤느 시칼파트+ 리페어 크림**  
   - 진정과 보습에 좋은 제품으로 특히 건조하고 민감한 피부에 적합합니다.

4. **일리윤 세라마이드 아토 크림**  
   - 세라마이드 성분이 피부 장벽을 강화하고 깊은 보습을 유지해줍니다.

구매 시 본인의 피부 상태와 알러지 반응 여부를 고려하시고, 샘플 사용 후 선택하시길 권장합니다! :)

Routing Trace: NONE
  -> No way to know which role handled this query
------------------------------------------------------------

Query: 설화수 자음생 크림 가격 알려줘
Response:
설화수 자음생 크림의 가격은 일반적으로 60ml 제품이 약 28만 원에서 30만 원 사이로 판매됩니다. 정확한 가격은 구매처와 프로모션에 따라 다를 수 있으니, 원하시는 매장 또는 공식 온라인 스토어에서 확인해보시는 것을 추천드립니다.

Routing Trace: NONE
  -> No way to know which role handled this query
------------------------------------------------------------

Query: 교환 환불 정책 알려줘
Response:
저희 교환 및 환불 정책은 다음과 같습니다:

1. 

---
# SECTION 1b: BEFORE — OK Architecture Mirror

**고객 현재 아키텍처 재현:** `ChatCompletionAgent` + plugins + `invoke()`

OK의 `ok_sk_agent_flow` 구조를 mirror합니다:
- Supervisor agent가 3개 plugin을 가지고 있음
- `invoke()` 한 번에 routing + tool calling + 응답 생성이 모두 처리됨
- **Routing decision이 LLM call 안에 buried** → 별도 로깅 hook 없음

```
ChatCompletionAgent(kernel, supervisor_instructions)
    └── invoke(history)
        ├── LLM decides which plugin/tool to call   ◄── ROUTING HERE (no hook)
        ├── tool calls execute                       ◄── NO SEPARATE LOG
        └── response streams back                    ◄── COMBINED OUTPUT
```

| Routing Trace | Override | Logging Hook |
|:---:|:---:|:---:|
| ❌ None | ❌ No | ❌ No |


In [5]:
# Cell 6: Section 1b — Run OK pattern supervisor
from agents.before.blackbox_ok_pattern import run_ok_pattern_supervisor

ok_kernel = create_kernel()

queries_1b = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 1b: OK Architecture Mirror')
print('(ChatCompletionAgent + Plugins)')
print('=' * 60)

for q in queries_1b:
    print(f'\nQuery: {q}')
    # <<< ROUTING DECISION HAPPENS HERE — buried inside LLM call, no hook available >>>
    response = await run_ok_pattern_supervisor(q, ok_kernel)
    # <<< TOOL SELECTION — no separate logging, combined with response generation >>>
    print(f'Response:\n{response[:500]}')
    print(f'\nRouting Trace: NONE')
    print(f'  -> Tool calls happened but no routing-level visibility')
    # <<< This is why the customer cannot see which agent was selected or why >>>
    print('-' * 60)


SECTION 1b: OK Architecture Mirror
(ChatCompletionAgent + Plugins)

Query: 건성 피부에 좋은 크림 추천해줘
{
    "name": "execute_tool RecommendPlugin-recommend_product",
    "context": {
        "trace_id": "0xf4fb08aa623c448e2544377b544d5611",
        "span_id": "0x2dff14281d5f899c",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x5b89a0ffde0e3dbc",
    "start_time": "2026-04-13T14:50:14.562426Z",
    "end_time": "2026-04-13T14:50:14.562706Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "RecommendPlugin-recommend_product",
        "gen_ai.tool.call.id": "call_uBf03ryrtEg3iWTcOhsX6MwK",
        "gen_ai.tool.description": "\ud53c\ubd80 \ud0c0\uc785\uc774\ub098 \uace0\ubbfc\uc5d0 \ub9de\ub294 \uc0c1\ud488\uc744 \ucd94\ucc9c\ud569\ub2c8\ub2e4. \uc608: '\uac74\uc131 \ud53c\ubd80\uc5d0 \uc88b\uc740 \ud06c\ub9bc \ucd94\ucc9c\ud574\uc918'"
    },
    "events": 

---
# SECTION 2: AFTER — Transparent Routing

**`AgentGroupChat` + `KernelFunctionSelectionStrategy`로 라우팅을 분리합니다.**

핵심 변화:
- **LLM call #1** (Selection): 어떤 agent를 선택할지 결정 (routing only)
- **`result_parser`** 가 선택 결과를 structured JSON으로 로깅
- **LLM call #2** (Response): 선택된 agent가 실제 응답 생성
- **Override hook**: 잘못된 라우팅을 커스텀 로직으로 교정 가능
- **OpenTelemetry** span으로 Application Insights에 trace 전송

| Routing Trace | Override | Logging Hook | OTel |
|:---:|:---:|:---:|:---:|
| ✅ agent, reason, confidence | ✅ Yes | ✅ JSON | ✅ App Insights |


In [6]:
# Cell 8: Section 2 — Transparent routing queries
from agents.after.transparent_routing import build_transparent_routing_chat

after_kernel = create_kernel()

queries_2 = [
    '건성 피부에 좋은 크림 추천해줘',
    '설화수 자음생 크림 가격 알려줘',
    '교환 환불 정책 알려줘',
]

print('=' * 60)
print('SECTION 2: Transparent Routing')
print('(AgentGroupChat + KernelFunctionSelectionStrategy)')
print('=' * 60)

for q in queries_2:
    print(f'\nQuery: {q}')
    chat, agents = build_transparent_routing_chat(
        kernel=after_kernel, logger=logger, tracer=tracer, query=q,
    )
    await chat.add_chat_message(message=q)
    async for response in chat.invoke():
        if response.content and response.name:
            print(f'\n[{response.name}]: {response.content[:500]}')
    print(f'\nRouting Trace: See structured JSON log above')
    print('-' * 60)


SECTION 2: Transparent Routing
(AgentGroupChat + KernelFunctionSelectionStrategy)

Query: 건성 피부에 좋은 크림 추천해줘
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x48f31823477e2f3d1dc36f4ffe7fb104",
        "span_id": "0x60d47ad003f81f19",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-13T14:50:37.637771Z",
    "end_time": "2026-04-13T14:50:41.702526Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-13T14:50:41.7043

In [7]:
# Cell 9: Section 2 — Override demo with ambiguous query
ambiguous_query = '설화수 자음생 크림 건성 피부에 좋아?'

override_rules = {
    '설화수 자음생 크림 건성 피부에 좋아?': 'SearchAgent',
}

print('=' * 60)
print('SECTION 2: Override Demo — Ambiguous Query')
print('=' * 60)

# Without override
print(f'\n[Without Override]')
print(f'Query: {ambiguous_query}')
chat_no, _ = build_transparent_routing_chat(
    kernel=after_kernel, logger=logger, tracer=tracer,
    query=ambiguous_query,
)
await chat_no.add_chat_message(message=ambiguous_query)
async for resp in chat_no.invoke():
    if resp.content and resp.name:
        print(f'[{resp.name}]: {resp.content[:300]}')

# With override
print(f'\n[With Override] (override_applied: true)')
print(f'Query: {ambiguous_query}')
chat_ov, _ = build_transparent_routing_chat(
    kernel=after_kernel, logger=logger, tracer=tracer,
    query=ambiguous_query, override_rules=override_rules,
)
await chat_ov.add_chat_message(message=ambiguous_query)
async for resp in chat_ov.invoke():
    if resp.content and resp.name:
        print(f'[{resp.name}]: {resp.content[:300]}')

print(f'\nCompare logs: override_applied false -> true')
print('-' * 60)


SECTION 2: Override Demo — Ambiguous Query

[Without Override]
Query: 설화수 자음생 크림 건성 피부에 좋아?
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0x09d67e059a3403523956732b6219d5b9",
        "span_id": "0xd2caa227760e31d6",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": null,
    "start_time": "2026-04-13T14:52:03.349922Z",
    "end_time": "2026-04-13T14:52:06.294217Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-13T14:52:06.295439+00:00", "leve

---
# SECTION 3: AFTER+ — Self-Reflection

**Section 2 + explicit `reflect()` function으로 응답 품질을 검증합니다.**

Flow:
1. **Routing** → AgentGroupChat이 agent 선택 (Section 2와 동일)
2. **KB Retrieval** → 선택된 agent가 mock KB에서 데이터 조회
3. **Reflection** → `reflect()` 함수가 별도 LLM call로 응답 품질 평가
4. **Re-route (REPLACE)** → 품질 미달 시 다른 agent로 교체, 원래 응답 폐기

핵심: `reflect()`는 **SK framework hook에 의존하지 않는** 명시적 Python function  
→ SK / MAF / LangGraph 어디서든 동일하게 작동 (framework-portable)

| Routing | Reflection | Re-route | Framework Portable |
|:---:|:---:|:---:|:---:|
| ✅ Full | ✅ Explicit | ✅ REPLACE | ✅ SK/MAF/LangGraph |


In [8]:
# Cell 11: Section 3 — Setup
from agents.after.self_reflection_routing import reflect, ReflectionResult
from agents.after.transparent_routing import (
    build_transparent_routing_chat,
    RECOMMEND_AGENT, SEARCH_AGENT, POLICY_AGENT,
)

reflection_kernel = create_kernel()

print('Section 3 setup complete')
print('  reflect() is a plain Python async function')
print('  -> Zero SK framework hook dependency')
print('  -> Portable to MAF / LangGraph without change')
print('  Pattern: route -> respond -> reflect -> (re-route if needed) -> final')


Section 3 setup complete
  reflect() is a plain Python async function
  -> Zero SK framework hook dependency
  -> Portable to MAF / LangGraph without change
  Pattern: route -> respond -> reflect -> (re-route if needed) -> final


In [9]:
# Cell 12: Section 3 — Ambiguous query with self-reflection
from semantic_kernel.contents import ChatHistory

ambiguous_query_3 = '설화수 자음생 크림 건성 피부에 좋아?'

print('=' * 60)
print('SECTION 3: Self-Reflection — Full Cycle')
print('=' * 60)
print(f'\nQuery: {ambiguous_query_3}')

# Step 1: Route via AgentGroupChat
with tracer.start_as_current_span('section3_full_cycle') as span:
    span.set_attribute('query', ambiguous_query_3)

    chat_3, agents_3 = build_transparent_routing_chat(
        kernel=reflection_kernel, logger=logger, tracer=tracer,
        query=ambiguous_query_3,
    )
    await chat_3.add_chat_message(message=ambiguous_query_3)

    first_agent_name = None
    first_response = None

    async for response in chat_3.invoke():
        if response.content and response.name:
            first_agent_name = response.name
            first_response = response.content
            print(f'\n[Step 1 - Routing] Selected: {first_agent_name}')
            print(f'[{first_agent_name}]: {first_response[:400]}')

    # Step 2: KB retrieval (mock)
    if first_agent_name == RECOMMEND_AGENT:
        kb_raw = rp.recommend_product(ambiguous_query_3)
    elif first_agent_name == SEARCH_AGENT:
        kb_raw = sp.search_product(ambiguous_query_3)
    else:
        kb_raw = pp.lookup_policy(ambiguous_query_3)

    kb_results = json.loads(kb_raw)
    kb_items = kb_results.get('recommendations', kb_results.get('results', kb_results.get('policies', [])))
    print(f'\n[Step 2 - KB Retrieval] {len(kb_items)} items from {first_agent_name}')

    # Step 3: Self-reflection
    print(f'\n[Step 3 - Reflection] Evaluating response quality...')
    reflection = await reflect(
        query=ambiguous_query_3,
        agent_name=first_agent_name or '',
        agent_response=first_response or '',
        kb_results=kb_items,
        kernel=reflection_kernel,
    )

    print(f'  intent_match:    {reflection.intent_match}')
    print(f'  sufficient_info: {reflection.sufficient_info}')
    print(f'  should_reroute:  {reflection.should_reroute}')
    print(f'  reroute_to:      {reflection.reroute_to}')
    print(f'  reason:          {reflection.reason}')

    # Warn if reflection LLM returned malformed JSON
    if 'PARSE_FAILED' in reflection.reason:
        print(f'\n⚠️ WARNING: Reflection LLM returned malformed JSON.')
        print(f'  Raw output: {reflection.reason}')
        print(f'  Keeping original response as fallback.')

    # Step 4: Re-route if needed (REPLACE pattern)
    if reflection.should_reroute and reflection.reroute_to:
        reroute_agent_name = reflection.reroute_to
        print(f'\n[Step 4 - REPLACE] Re-routing: {first_agent_name} -> {reroute_agent_name}')

        reroute_agent = agents_3.get(reroute_agent_name)
        if reroute_agent:
            reroute_history = ChatHistory()
            reroute_history.add_user_message(ambiguous_query_3)
            final_response = None
            async for msg in reroute_agent.invoke(reroute_history):
                # msg is AgentResponseItem — use str() to get text content
                text = str(msg)
                if text:
                    final_response = text
                    print(f'[Final - {reroute_agent_name}]: {final_response[:400]}')
            action = 'REPLACE'
            final_agent = reroute_agent_name
        else:
            print(f'Agent {reroute_agent_name} not found, keeping original')
            final_response = first_response
            action = 'KEEP'
            final_agent = first_agent_name
    else:
        print(f'\n[Step 4] Reflection passed — keeping original response')
        final_response = first_response
        action = 'KEEP'
        final_agent = first_agent_name

    # Log self-reflection event
    log_self_reflection(
        logger,
        query=ambiguous_query_3,
        agent=first_agent_name or '',
        kb_results_count=len(kb_items),
        reflection={
            'intent_match': reflection.intent_match,
            'sufficient_info': reflection.sufficient_info,
            'should_reroute': reflection.should_reroute,
            'reroute_to': reflection.reroute_to,
            'reason': reflection.reason,
        },
        action=action,
        final_agent=final_agent,
    )
    span.set_attribute('reflection.action', action)
    span.set_attribute('reflection.final_agent', final_agent or '')

print('\n' + '=' * 60)
print('Full Decision Trace:')
print(f'  1. Routing    -> {first_agent_name}')
print(f'  2. KB         -> {len(kb_items)} items retrieved')
print(f'  3. Reflection -> should_reroute={reflection.should_reroute}')
print(f'  4. Action     -> {action} (final agent: {final_agent})')
print('=' * 60)


SECTION 3: Self-Reflection — Full Cycle

Query: 설화수 자음생 크림 건성 피부에 좋아?
{
    "name": "execute_tool routing_selection",
    "context": {
        "trace_id": "0xc6977d93cda2c486d6a86940d32defec",
        "span_id": "0x4751791c7013217e",
        "trace_state": "[]"
    },
    "kind": "SpanKind.INTERNAL",
    "parent_id": "0x6f922cdc28024dbd",
    "start_time": "2026-04-13T14:53:19.198847Z",
    "end_time": "2026-04-13T14:53:23.712415Z",
    "status": {
        "status_code": "UNSET"
    },
    "attributes": {
        "gen_ai.operation.name": "execute_tool",
        "gen_ai.tool.name": "routing_selection"
    },
    "events": [],
    "links": [],
    "resource": {
        "attributes": {
            "telemetry.sdk.language": "python",
            "telemetry.sdk.name": "opentelemetry",
            "telemetry.sdk.version": "1.41.0",
            "service.name": "routing-observability-demo"
        },
        "schema_url": ""
    }
}
{"timestamp": "2026-04-13T14:53:23.713278+00:00", "level": "I

---
# SUMMARY

## Before vs After

| 항목 | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ |
|------|:---:|:---:|:---:|:---:|
| **Routing 분리** | ❌ | ❌ | ✅ | ✅ |
| **Routing Trace** | ❌ | ❌ | ✅ agent, reason, confidence | ✅ + reflection |
| **Override Hook** | ❌ | ❌ | ✅ result_parser | ✅ |
| **Self-Reflection** | ❌ | ❌ | ❌ | ✅ explicit reflect() |
| **Re-route (REPLACE)** | ❌ | ❌ | ❌ | ✅ |
| **OTel Tracing** | ❌ | ❌ | ✅ | ✅ |
| **Framework Portable** | — | — | SK only | ✅ SK/MAF/LangGraph |

## Key Takeaways

1. **SK는 이미 투명한 라우팅을 공식 지원합니다** — `KernelFunctionSelectionStrategy`를 활용하면 기존 코드 구조를 크게 변경하지 않고도 라우팅 가시성을 확보할 수 있습니다.
2. **라우팅과 응답 생성을 분리하면 운영이 쉬워집니다** — 로깅, 오버라이드, 모니터링이 자연스럽게 따라옵니다.
3. **Self-reflection은 framework에 종속되지 않습니다** — explicit function으로 구현하면 어디로든 이식 가능합니다.


In [10]:
# Cell 14: Summary — comparison table + App Insights KQL
from IPython.display import display, Markdown

comparison = """
## Side-by-Side Comparison

| Feature | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ |
|---------|:---:|:---:|:---:|:---:|
| LLM Calls / Query | 1 | 1 (+ tool calls) | 2 (select + respond) | 3~4 (select + respond + reflect [+ re-route]) |
| Routing Visible | No | No | Yes | Yes |
| Agent / Reason / Confidence | — | — | JSON logged | JSON logged |
| Override Support | No | No | result_parser | result_parser |
| Quality Check | No | No | No | reflect() |
| Re-route on Failure | No | No | No | REPLACE |
| OTel / App Insights | No | No | Yes | Yes |
| SK API Used | AzureChatCompletion | ChatCompletionAgent | AgentGroupChat | AgentGroupChat + reflect() |
"""
display(Markdown(comparison))

kql = """
## Application Insights — Routing Trace Query (KQL)

```kql
-- Option A: traces 테이블 (OTel span이 trace로 매핑된 경우 — 일반적)
traces
| where timestamp > ago(24h)
| where customDimensions has "routing.agent"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
-- Option B: dependencies 테이블 (span이 dependency로 매핑된 경우)
dependencies
| where timestamp > ago(24h)
| where name == "routing_decision"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
// Self-reflection re-routes
traces
| where timestamp > ago(24h)
| where customDimensions has "reflection.action"
| extend action = tostring(customDimensions["reflection.action"])
| extend final_agent = tostring(customDimensions["reflection.final_agent"])
| where action == "REPLACE"
| project timestamp, action, final_agent
```
"""
display(Markdown(kql))

print('\nDemo complete!')



## Side-by-Side Comparison

| Feature | 1a: Black Box | 1b: OK Mirror | 2: After | 3: After+ |
|---------|:---:|:---:|:---:|:---:|
| LLM Calls / Query | 1 | 1 (+ tool calls) | 2 (select + respond) | 3~4 (select + respond + reflect [+ re-route]) |
| Routing Visible | No | No | Yes | Yes |
| Agent / Reason / Confidence | — | — | JSON logged | JSON logged |
| Override Support | No | No | result_parser | result_parser |
| Quality Check | No | No | No | reflect() |
| Re-route on Failure | No | No | No | REPLACE |
| OTel / App Insights | No | No | Yes | Yes |
| SK API Used | AzureChatCompletion | ChatCompletionAgent | AgentGroupChat | AgentGroupChat + reflect() |



## Application Insights — Routing Trace Query (KQL)

```kql
-- Option A: traces 테이블 (OTel span이 trace로 매핑된 경우 — 일반적)
traces
| where timestamp > ago(24h)
| where customDimensions has "routing.agent"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
-- Option B: dependencies 테이블 (span이 dependency로 매핑된 경우)
dependencies
| where timestamp > ago(24h)
| where name == "routing_decision"
| extend agent = tostring(customDimensions["routing.agent"])
| extend reason = tostring(customDimensions["routing.reason"])
| extend confidence = todouble(customDimensions["routing.confidence"])
| extend override = tobool(customDimensions["routing.override_applied"])
| project timestamp, agent, reason, confidence, override
| order by timestamp desc
```

```kql
// Self-reflection re-routes
traces
| where timestamp > ago(24h)
| where customDimensions has "reflection.action"
| extend action = tostring(customDimensions["reflection.action"])
| extend final_agent = tostring(customDimensions["reflection.final_agent"])
| where action == "REPLACE"
| project timestamp, action, final_agent
```



Demo complete!
